# 3b. Contract negotiation through the BFF routes

The admin GUI negotiates through `/dsp/current/negotiations/bff-rpc/`: the same eunomia bodies
and answers as the RPCs of `contract_negotiation.ipynb`, each sending one message. The BFF covers
the steps a user decides on (open, accept, agree, terminate); the Consumer's verification and the
Provider's finalization go through their own RPCs.

This notebook shows the two shortest ways to an agreement:

| Part | Who opens | Messages | |
|---|---|---|---|
| I | Consumer | request -> agreement -> verification -> finalization | The Provider agrees on the request as is |
| II | Provider | offer -> acceptance -> agreement -> verification -> finalization | The Consumer accepts the Provider's offer |

**Runs on its own.** It leaves `agreement_id` (from Part II) and records
`agreement_source = ("bff", target_id)`.

## 1. Prerequisites

The guard populates a catalog only when no catalog notebook has run in this kernel; it runs
`setup_catalog_policies.ipynb`, which in turn onboards the participants if needed. Run
`setup_catalog_policies_oauth.ipynb` first to negotiate on its dataset instead.

In [ ]:
if not globals().get("catalog_setup"):
    %run ./setup_catalog_policies.ipynb

### How the negotiation RPCs work

Neither participant calls the peer's DSP endpoints directly. Each one asks **its own agent** to
send the next DSP message through an RPC under `/dsp/current/negotiations/rpc/`. The body is
eunomia JSON-LD:

```json
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcNegotiation<Kind>Message",
  "consumerPid": "...", "providerPid": "...",
  "offer": { ... }
}
```

The agent reads it through RDF, builds the DSP message, checks it against the negotiation state
machine and its own role, sends it with its peer token and stores the result once the peer
accepts it. Every RPC answers an `RpcNegotiationResult`:
- `request`: the RPC body as received.
- `response`: the peer's answer (its `ContractNegotiation` ACK for an opening message).
- `negotiationAgentModel`: the caller's own process as it stands, with `state`, `identifiers`,
  `messages`, `offers` and, once agreed, `agreement`.

An opening RPC answers 201, the rest 200. Errors answer a `ContractNegotiationError`: 400 for a
broken rule, 404 for an unknown negotiation, 409 for a stale or repeated message, 502 when the
peer refused it. A continuation names the negotiation by `providerPid` or `consumerPid`; these
notebooks send both.

In [ ]:
def negotiation_step(response, expected_state):
    """Prints an RpcNegotiationResult and checks the caller's process reached expected_state."""
    answer = show(response)
    process = answer["negotiationAgentModel"]
    assert process["state"] == expected_state, f"expected {expected_state}, got {process['state']}"
    print(f"\n==> {process['state']}")
    return process

# Part I: the Consumer requests, the Provider agrees

## 2. Contract request (`bff-rpc/setup-request-init`, Consumer -> Provider)

The Consumer opens the negotiation on the published offer. DSP 8.2.2.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationRequestInitMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/bff-rpc/setup-request-init", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "REQUESTED")
cn_consumer_id = consumer_process["identifiers"]["consumerPid"]
cn_provider_id = consumer_process["identifiers"]["providerPid"]

## 3. Agreement (`bff-rpc/setup-agreement`, Provider -> Consumer)

The Provider agrees on the requested offer directly, without a counter-offer. DSP 8.3.5.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAgreementMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/bff-rpc/setup-agreement", headers=json_header, json=payload)
provider_process = negotiation_step(response, "AGREED")

## 4. Verification (`rpc/setup-verification`, Consumer -> Provider)

DSP 8.2.5.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationVerificationMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-verification", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "VERIFIED")

## 5. Finalization (`rpc/setup-finalization`, Provider -> Consumer)

DSP 8.3.6.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationFinalizationMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-finalization", headers=json_header, json=payload)
provider_process = negotiation_step(response, "FINALIZED")

## 6. Agreement and final state on both sides

The agreement is the last offer both sides accepted, now with `assigner` (Provider) and `assignee`
(Consumer). Both agents keep the same agreement id; it is the `agreementId` a transfer request
names. The management API of each agent shows its own view of the process.

In [ ]:
agreement = provider_process["agreement"]
agreement_id = agreement["id"]
print(json.dumps(agreement, indent=2))

for name, agent_url, process in [("Consumer", data_space_consumer, consumer_process), ("Provider", data_space_provider, provider_process)]:
    view = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-processes/{process['id']}", headers=json_header).json()
    print(f"{name}: {view['state']}, agreement {view['agreement']['id']}")

# Part II: the Provider offers, the Consumer accepts

## 7. Offer (`bff-rpc/setup-offer-init`, Provider -> Consumer)

The Provider opens the negotiation by sending an offer to the Consumer. DSP 8.3.3. The roles of
the addresses swap with respect to Part I: `associatedAgentPeer` is the Consumer,
`consumerAddress` its DSP base URL, and `callbackAddress` the Provider's own.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationOfferInitMessage",
    "associatedAgentPeer": data_space_consumer_participant_id,
    "consumerAddress": data_space_consumer_dsp_endpoint,
    "callbackAddress": data_space_provider_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/bff-rpc/setup-offer-init", headers=json_header, json=payload)
provider_process = negotiation_step(response, "OFFERED")
cn_consumer_id = provider_process["identifiers"]["consumerPid"]
cn_provider_id = provider_process["identifiers"]["providerPid"]

## 8. Acceptance (`bff-rpc/setup-acceptance`, Consumer -> Provider)

DSP 8.2.4.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAcceptanceMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/bff-rpc/setup-acceptance", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "ACCEPTED")

## 9. Agreement (`bff-rpc/setup-agreement`, Provider -> Consumer)

DSP 8.3.5.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAgreementMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/bff-rpc/setup-agreement", headers=json_header, json=payload)
provider_process = negotiation_step(response, "AGREED")

## 10. Verification and finalization (`rpc/`)

The Consumer verifies (DSP 8.2.5) and the Provider finalizes (DSP 8.3.6).

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationVerificationMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-verification", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "VERIFIED")

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationFinalizationMessage",
    "consumerPid": cn_consumer_id,
    "providerPid": cn_provider_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-finalization", headers=json_header, json=payload)
provider_process = negotiation_step(response, "FINALIZED")

## 11. Agreement and final state on both sides

The agreement is the last offer both sides accepted, now with `assigner` (Provider) and `assignee`
(Consumer). Both agents keep the same agreement id; it is the `agreementId` a transfer request
names. The management API of each agent shows its own view of the process.

In [ ]:
agreement = provider_process["agreement"]
agreement_id = agreement["id"]
print(json.dumps(agreement, indent=2))

for name, agent_url, process in [("Consumer", data_space_consumer, consumer_process), ("Provider", data_space_provider, provider_process)]:
    view = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-processes/{process['id']}", headers=json_header).json()
    print(f"{name}: {view['state']}, agreement {view['agreement']['id']}")

In [ ]:
# Marker read by the guard of bff_push_flow.ipynb
agreement_source = ("bff", target_id)
print(f"agreement_id = {agreement_id}")